# 🐍 Clase 18: El modelo — desmontar una ilusión 🎭

**Objetivo de la clase:** entrenar tu primer modelo… y aprender a NO creerle: baseline,
matriz de confusión, métrica balanceada, dispersión, la fuga 💸 y la prueba de permutación.
Al final, tu experimento queda **registrado** y entra a `main` por tu primer **pull request**.

## 🗺️ Mapa de la clase

| Etapa | Tema | El número (verificado) |
|---|---|---|
| **1** | Los datos al modelo | X: (140, 5) · y: 109/31 · **a vencer: 0,779** |
| **2** | La trampa del 77 % | LR de fábrica: **0,769 ± 0,033** — pierde contra no hacer nada |
| **3** | La matriz de confusión | errores detectados ≈ **0** |
| **4** | La métrica balanceada | el modelo honesto: **0,647 ± 0,090** |
| **5** | El ruido de n=140 | ese **± 0,090** ES el mensaje |
| **6** | La fuga 💸 | +`post_retro` → **0,697** («mejoró»… no) · 24,5 vs 19,3 Hz |
| **7** | ¿Azar? Permutación | nulo 0,505 ± 0,049 → **p = 0,005** |
| **8** | Registrar y entregar | `resultados.json` → rama → **PR → merge** |

## 🔗 ¿Recuerdas dónde nos quedamos?

El martes formulaste la pregunta y anotaste el número a vencer. Y quedó un pagaré: la
columna `post_retro`, medida DESPUÉS de la recompensa, *huele raro*. **Hoy se cobra, con
datos.** El protocolo es fijo para todo el salón — semilla 0, validación 5×20, balanced
accuracy — así **tu número y el del vecino serán idénticos**, y ambos iguales a los del
profe.

In [ ]:
# Punto de control · ¿está lista la mesa? ✅
import sqlite3
from pathlib import Path

FALTA = []
if not Path("neurona.sqlite").exists():
    FALTA.append("neurona.sqlite (cuaderno 17, Etapa 2: python scripts/construir_bd.py)")
else:
    con = sqlite3.connect("neurona.sqlite")
    tablas = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if "tasas" not in tablas:
        FALTA.append("la tabla `tasas` (cuaderno 17, Etapa 4)")
try:
    import sklearn  # noqa
except ImportError:
    FALTA.append("scikit-learn (Etapa 1 de HOY: pip install scikit-learn)")

print("✅ Todo listo: a desmontar la ilusión." if not FALTA else "⏳ Falta:\n   - " + "\n   - ".join(FALTA))

---
# 🟢 Etapa 1 — Los datos al modelo

*📽 Vienes de MOD-01.*

**1a · La dependencia nueva** (el rito ya lo conoces): en la terminal `(.venv)`:

```
pip install scikit-learn
```

añade `scikit-learn>=1.3` al final de `requirements.txt`, y commit:
`git commit -m "El proyecto aprende a modelar: entra scikit-learn"` → push.

**1b · X e y desde tu base.** Corre la celda — además fija el **protocolo del salón**.

In [ ]:
# X, y, y el protocolo FIJO (así tu número == el del vecino == el del profe)
import numpy as np, pandas as pd

try:
    from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
    from sklearn.linear_model import LogisticRegression
    from sklearn.dummy import DummyClassifier
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    EPOCAS_HONESTAS = ["basal", "P1", "demora1", "P2", "demora2"]   # todo ANTES de la respuesta
    tasas = pd.read_sql("SELECT * FROM tasas", con)
    X = tasas[EPOCAS_HONESTAS].to_numpy()
    y = tasas["acierto"].to_numpy().astype(int)

    SEED = 0
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=SEED)  # 100 mediciones

    print(f"X: {X.shape}   y: {y.sum()} aciertos / {(y==0).sum()} errores")
    print(f"El número a vencer (clase mayoritaria): {y.mean():.3f}")
except ImportError:
    print("⏳ Falta scikit-learn — Etapa 1a.")
except NameError:
    print("⏳ Corre primero el punto de control.")

*▶ Volvemos a las diapositivas: MOD-02 — el baseline.*

---
# 🟢 Etapa 2 — La trampa del 77 %

In [ ]:
# El baseline medido… y la primera ilusión
try:
    dummy = cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=cv, scoring="accuracy")
    print(f"[baseline]  decir siempre «acierto» ......... {dummy.mean():.3f}")

    lr_fabrica = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    acc = cross_val_score(lr_fabrica, X, y, cv=cv, scoring="accuracy")
    print(f"[modelo]    regresión logística de fábrica ... {acc.mean():.3f} ± {acc.std():.3f}")
    print()
    if acc.mean() < dummy.mean():
        print("🐞 Detective: el «modelo entrenado» quedó POR DEBAJO de no hacer nada.")
        print("   Predice 🔮 antes de la Etapa 3: ¿qué está haciendo por dentro?")
except NameError:
    print("⏳ Corre la Etapa 1.")

*▶ Volvemos a las diapositivas: MOD-03 — la matriz.*

---
# 🟢 Etapa 3 — El microscopio: tu matriz de confusión

In [ ]:
# ¿Qué hace por dentro el modelo del 77 %? La matriz lo delata
try:
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import confusion_matrix

    pred = cross_val_predict(lr_fabrica, X, y, cv=StratifiedKFold(5))
    m = confusion_matrix(y, pred, labels=[1, 0])
    print("                 predijo ACIERTO   predijo ERROR")
    print(f"real ACIERTO(109)      {m[0,0]:4d}             {m[0,1]:4d}")
    print(f"real ERROR   (31)      {m[1,0]:4d}             {m[1,1]:4d}   ← la casilla que importa")
    print(f"\n🔎 El veredicto: detectó {m[1,1]} de los 31 errores. No aprendió: apostó a la mayoría.")
except NameError:
    print("⏳ Corre las etapas 1 y 2.")

*▶ Volvemos a las diapositivas: MOD-04 — la métrica balanceada.*

---
# 🟡 Etapa 4 — Peor número, mejor modelo

In [ ]:
# El modelo honesto: clases pesadas + métrica balanceada (el protocolo completo)
try:
    modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
    scores = cross_val_score(modelo, X, y, cv=cv, scoring="balanced_accuracy")
    print(f"balanced accuracy (100 mediciones): {scores.mean():.3f} ± {scores.std():.3f}")
    print("   → contra 0,50 de azar (la balanceada no se deja engañar por el 78/22)")
    print("   → compara con tu vecino: DEBE ser idéntico — esa es la semilla trabajando")
except NameError:
    print("⏳ Corre la Etapa 1.")

*▶ Volvemos a las diapositivas: MOD-05 — el ± es el mensaje.*

---
# 🟡 Etapa 5 — Mira el ruido con tus propios ojos

In [ ]:
# Las 100 mediciones no son un número: son una NUBE
import matplotlib.pyplot as plt

try:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.boxplot(scores, vert=False, widths=0.5)
    ax.scatter(scores, np.random.default_rng(0).uniform(0.85, 1.15, len(scores)), alpha=0.35, s=14)
    ax.axvline(0.5, ls="--", color="gray"); ax.text(0.502, 1.35, "azar", color="gray")
    ax.set_xlabel("balanced accuracy"); ax.set_yticks([])
    ax.set_title(f"la MISMA neurona, 100 particiones: {scores.mean():.3f} ± {scores.std():.3f}")
    plt.tight_layout(); plt.show()
    print("Una corrida «suertuda» y una «mala» viven en la misma nube. Por eso: media ± desviación, siempre.")
except NameError:
    print("⏳ Corre la Etapa 4.")

*▶ Volvemos a las diapositivas: MOD-06 — la fuga 💸.*

---
# 🔵 Etapa 6 — Se cobra el pagaré: produce la fuga (y desmóntala)

> 🤖 **Cameo de IA (2 min):** pega a tu IA la lista de features
> `[basal, P1, demora1, P2, demora2, post_retro]` con una línea de contexto («predecir el
> acierto del mono en cada ensayo») y pregúntale: *«¿ves fuga de información en estas
> features?»*. Anota en `docs/reporte.md` qué respondió — y si la encontró antes que tú.

In [ ]:
# La época prohibida entra al modelo… y «mejora»
try:
    X_fuga = tasas[EPOCAS_HONESTAS + ["post_retro"]].to_numpy()
    s_fuga = cross_val_score(modelo, X_fuga, y, cv=cv, scoring="balanced_accuracy")
    print(f"con post_retro: {s_fuga.mean():.3f} ± {s_fuga.std():.3f}   (antes: {scores.mean():.3f})  → «¡mejoró!» 🎉…")
    print()
    hz_err = tasas.loc[tasas.acierto == 0, "post_retro"].mean()
    hz_ok  = tasas.loc[tasas.acierto == 1, "post_retro"].mean()
    print(f"…¿por qué? Tras la retroalimentación la neurona dispara distinto:")
    print(f"   {hz_err:.1f} Hz tras ERRORES   vs   {hz_ok:.1f} Hz tras ACIERTOS")
    print("\n🔎 El veredicto: esa ventana no PREDICE el acierto — lo REPORTA. Es fuga.")
    print("   (Y la fuga silenciosa que NO cometimos: el scaler vive DENTRO del pipeline,")
    print("    así se ajusta solo con el fold de entrenamiento.)")
except NameError:
    print("⏳ Corre las etapas 1 y 4.")

*▶ Volvemos a las diapositivas: MOD-07 — la permutación.*

---
# 🔵 Etapa 7 — Los 200 mundos barajados *(⏱ tarda ~1 min: respira)*

In [ ]:
# ¿Y si todo fuera azar? Baraja y 200 veces y compara
try:
    from sklearn.model_selection import permutation_test_score
    score, permutados, p = permutation_test_score(
        modelo, X, y, cv=cv, scoring="balanced_accuracy",
        n_permutations=200, random_state=SEED, n_jobs=-1)
    print(f"score real: {score:.3f}    mundos barajados: {permutados.mean():.3f} ± {permutados.std():.3f}    p = {p:.3f}")

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(permutados, bins=25, alpha=0.75, label="200 mundos sin señal")
    ax.axvline(score, color="crimson", lw=2.5, label=f"tu score real ({score:.3f})")
    ax.set_xlabel("balanced accuracy"); ax.legend(); plt.tight_layout(); plt.show()
    print("🔎 Veredicto: «sabía un poco — sobre todo durante P2». Débil, pero real: p = %.3f." % p)
except NameError:
    print("⏳ Corre las etapas 1 y 4.")

*▶ Volvemos a las diapositivas: MOD-08 (registrar) y GIT-04 (ramas y PR).*

---
# 🟣 Etapa 8 — Registra el experimento y entrégalo como profesional

**8a · La rama.** En la terminal (¡ANTES de escribir el archivo!):

```
git switch -c feature/modelo
```

**8b · El registro.** Corre la celda: escribe `resultados/resultados.json` con tu hash.

In [ ]:
# resultados.json — la idea de MLflow, a mano (y por eso, entendida)
import json, subprocess
from datetime import date

try:
    try:
        commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
    except Exception:
        commit = "sin-git"
    resultados = {
        "pregunta": "¿la neurona sabía? — predecir acierto/error desde tasas por época",
        "fecha": date.today().isoformat(), "commit": commit,
        "protocolo": {"modelo": "LogisticRegression(balanced) + StandardScaler",
                      "validacion": "RepeatedStratifiedKFold(5x20)",
                      "metrica": "balanced_accuracy", "semilla": SEED},
        "datos": {"n": len(y), "aciertos": int(y.sum()), "errores": int((y == 0).sum())},
        "resultados": {"honesto": [round(scores.mean(), 3), round(scores.std(), 3)],
                       "con_fuga_NO_USAR": [round(s_fuga.mean(), 3), round(s_fuga.std(), 3)],
                       "permutacion": {"nulo": [round(permutados.mean(), 3), round(permutados.std(), 3)],
                                        "p": round(float(p), 3)}},
        "veredicto": "señal débil y real (p<0.01), concentrada en P2; con n=31 errores, ±0.09 manda",
    }
    Path("resultados").mkdir(exist_ok=True)
    Path("resultados/resultados.json").write_text(json.dumps(resultados, indent=2, ensure_ascii=False))
    print("✔ resultados/resultados.json — con tu commit:", commit)
    print(json.dumps(resultados["resultados"], indent=2))
except NameError:
    print("⏳ Necesita las etapas 4, 6 y 7 corridas.")

**8c · El pull request — tu primera entrega profesional:**

```
git add resultados/ requirements.txt
git commit -m "Closes #5: el experimento, registrado y reproducible"
git push -u origin feature/modelo
```

*(abre antes el issue #5: «modelo · registrar el experimento»)* — y en la web:
**Compare & pull request** → lee TU diff (¡es tu ciencia!) → **Merge pull request**.

**Debes ver:** tu `main` con el commit de merge, la rama fusionada, y el issue #5 cerrado.
Ese ciclo —rama → PR → merge— es exactamente como entra el código a los proyectos reales.

---
## ✅ Checkpoint de la clase 🔮

1. El modelo de fábrica sacó 0,769 de *accuracy*. ¿Por qué eso era MALA noticia?
2. ¿Qué pregunta responde la matriz de confusión que la exactitud esconde?
3. ¿Por qué la balanceada usa 0,50 como azar, haya o no desbalance?
4. Tu compañera reporta 0,74 «porque le salió mejor con otra semilla». ¿Qué le dices, con MOD-05 en la mano?
5. Da un ejemplo de fuga en TU área (no el de la clase) — ¿qué información «del futuro» se colaría?
6. ¿Qué demuestra (y qué NO demuestra) p = 0,005 en la permutación?

> 💬 **Para tu reporte (Clase 4):** un titular dice *«Neurona de la corteza premotora
> predice las decisiones del mono»* citando un modelo como el nuestro. Escribe los 3–5
> renglones que le mandarías al periodista: qué es cierto, qué está inflado, y qué papel
> juegan n=31, el ±0,09 y la fuga. *(A `docs/reporte.md`, con tus usos de IA declarados.)*

---
# 🏆 Proyecto integrador — el cierre

Rúbrica: **funciona 70 % · comentado 15 % · discusión 15 %**.

**Hoy sales con:** el protocolo completo corrido y ENTENDIDO · `resultados.json` con tu
hash · tu primer **PR mergeado** · el issue #5 cerrado.

**Tarea (al martes 13):** `docs/reporte.md` completo — las 💬 de las cuatro clases más tus
declaraciones de IA. Es el 15 % de discusión de TODO el proyecto: trátalo como se merece.

---
# 🔭 ¿Qué sigue?

El martes 13 cierra el módulo (formato por confirmar con el profe titular). Y quedan tres
puertas abiertas un dedo 🚪: **Docker** (viste la demo — tu proyecto corriendo idéntico en
cualquier máquina), **PyTorch** (cuando tengas n de verdad) y **MLflow** (tu
`resultados.json`, industrializado). La ciencia de datos que viste esta semana —baseline,
honestidad, registro— es la que vale en cualquier laboratorio. El resto son herramientas. 🧠